In [7]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [8]:
# 0단계: 문서 로드
loader = PyPDFLoader('../data/콘텐츠분쟁해결_사례.pdf')
documents = loader.load()
print("페이지 수:", len(documents))

# 1단계: 문서 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        #  법률 사례는 1200-1800자 권장
    chunk_overlap=300,      #  사례 맥락 보존을 위해 200-400자
    separators=[
        "\n【사건개요】",     #  법률 문서 섹션 구분자
        "\n【쟁점사항】",     #  쟁점 부분 구분
        "\n【처리경위】",     #  처리 과정 구분
        "\n【처리결과】",     #  결과 부분 구분
        "\n■", "\n\n", "\n", ".", " ", ""
    ]
)

texts = loader.load_and_split(text_splitter=text_splitter)
print("청크 수:", len(texts))

페이지 수: 109
청크 수: 104


In [9]:
# 2단계: 임베딩 모델 설정
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# 벡터 저장소 생성 및 저장
vectorstore = FAISS.from_documents(texts, embeddings)
vectorstore.save_local("../db/content_dispute_faiss")

# 3단계: 검색기 설정
retriever = vectorstore.as_retriever(
    search_type="similarity",        #  또는 "mmr" (다양성 고려시)
    search_kwargs={"k": 5}          #  상위 5개 관련 사례 검색
)
print("검색기 설정 완료")

검색기 설정 완료


In [10]:
# 4단계: LLM 설정
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.2
    )
print(llm.model_name)

solar-pro3


In [11]:
# 5단계: 법률 자문 프롬프트 작성
prompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요
2. 관련 법령이나 조항이 있다면 명시하세요
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요
4. 실무적 해결방안을 단계별로 제시하세요
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요

전문 법률 조언:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
print("프롬프트 설정 완료")

프롬프트 설정 완료


In [13]:
def format_docs(docs):
    """Format retrieved docs as context string"""
    return "\n\n".join(doc.page_content for doc in docs)

# 6단계: LCEL 체인 생성
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("LCEL RAG chain 설정 완료")

LCEL RAG chain 설정 완료


In [14]:
# 8단계: 분쟁 유형 분류 함수 (선택)
def classify_dispute_type(query):
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]
    
    query_lower = query.lower()
    
    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"

# 7단계: 테스트 질문 작성
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"
]

for i, question in enumerate(test_questions, 1):
    print(f"[질문 {i}] ({classify_dispute_type(question)}) {question}")
    try:
        answer = qa_chain.invoke(question)
        print(answer)
    except Exception as e:
        print(f"오류: {e}")
    print("-" * 50)

[질문 1] (게임) 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?
### 온라인 게임 시스템 오류로 인한 아이템 소멸 관련 법률 조언

#### 1. **제시된 사례를 근거로 한 분석**
- **2006년 사례**: 프로그램 오류로 인한 아이템 소멸 시, 게임회사가 복구 가능성을 인정하고 복구 처리한 사례가 있습니다. 이는 "프로그램 오류"가 명확히 확인된 경우 회사가 책임을 인정한 것으로 보입니다.
- **2009년 사례**: 시스템 오류로 인한 게임머니 소실 주장 시, 회사는 "시스템 오류 발생 여부"를 엄격히 검토하여 사용자 측의 통신회선 불안정 또는 PC 오류로 판단했습니다. 이 경우 복구가 거부되었습니다.
- **계정 명의 문제**: 2006년 사례에서 계정 명의자가 아닌 경우 복구가 거부된 사례가 있습니다. 게임회사는 계정 명의자만을 법적 권리자로 인정하는 경향이 있습니다.

#### 2. **관련 법령 및 약관**
- **민법 제250조**: 도품·유실물에 대한 특례 (단, 금전은 제외)
- **민법 제251조**: 선의의 양수인에 대한 특례
- **게임회사 이용약관**: 일반적으로 "계정 및 아이템의 소유권은 회사에 귀속"되며, "계정 공유·현금거래 금지" 조항이 포함되어 있습니다.
- **전자상거래법**: 온라인 콘텐츠 제공자의 책임 범위 (시스템 오류로 인한 서비스 장애 시)

#### 3. **실무적 해결방안 (단계별)**
**1단계: 증거 확보**
- 시스템 오류 발생 시점, 아이템 소멸 로그, 오류 메시지 스크린샷 등 객관적 증거 수집
- 계정 명의 확인 (명의자 본인임을 증명할 수 있는 서류 준비)

**2단계: 게임회사에 공식 요청**
- 이용약관 제11조(이용자의 의무) 및 제12조(서비스 중단 등) 조항을 근거로 복구 요청
- "프로그램 오류" 또는 "시스템 장애"로 인한 피해임을 명시적으로 주장

**3단계: 한국소비자원 분쟁조정 신청**
- 2006년 사례